In [6]:
import http.client
import json

conn = http.client.HTTPSConnection("sportapi7.p.rapidapi.com")

headers = {
    'x-rapidapi-key': "2b277abcd2msh0e5627048810020p119057jsn4412b05235c9",
    'x-rapidapi-host': "sportapi7.p.rapidapi.com"
}

conn.request("GET", "/api/v1/tournament/1/seasons", headers=headers)

res = conn.getresponse()
# Volg redirect handmatig
data = res.read()
result = json.loads(data.decode("utf-8"))

for season in result.get('seasons', []):
    print(f"Seizoen: {season.get('name')} | ID: {season.get('id')}")

Seizoen: Premier League 25/26 | ID: 76986
Seizoen: Premier League 24/25 | ID: 61627
Seizoen: Premier League 23/24 | ID: 52186
Seizoen: Premier League 22/23 | ID: 41886
Seizoen: Premier League 21/22 | ID: 37036
Seizoen: Premier League 20/21 | ID: 29415
Seizoen: Premier League 19/20 | ID: 23776
Seizoen: Premier League 18/19 | ID: 17359
Seizoen: Premier League 17/18 | ID: 13380
Seizoen: Premier League 16/17 | ID: 11733
Seizoen: Premier League 15/16 | ID: 10356
Seizoen: Premier League 14/15 | ID: 8186
Seizoen: Premier League 13/14 | ID: 6311
Seizoen: Premier League 12/13 | ID: 4710
Seizoen: Premier League 11/12 | ID: 3391
Seizoen: Premier League 10/11 | ID: 2746
Seizoen: Premier League 09/10 | ID: 2139
Seizoen: Premier League 08/09 | ID: 1544
Seizoen: Premier League 07/08 | ID: 581
Seizoen: Premier League 06/07 | ID: 4
Seizoen: Premier League 05/06 | ID: 3
Seizoen: Premier League 04/05 | ID: 2
Seizoen: Premier League 03/04 | ID: 1
Seizoen: Premier League 02/03 | ID: 46
Seizoen: Premier Lea

In [13]:
import requests
import json

headers = {
    "x-rapidapi-key": "2b277abcd2msh0e5627048810020p119057jsn4412b05235c9",
    "x-rapidapi-host": "sofascore.p.rapidapi.com"
}

seizoenen = [
    {"naam": "Premier League 17/18", "id": 13380},
    {"naam": "Premier League 16/17", "id": 11733},
    {"naam": "Premier League 15/16", "id": 10356},
    {"naam": "Premier League 14/15", "id": 8186},
]

alle_wedstrijden = []

for seizoen in seizoenen:
    page = 0
    while True:
        url = "https://sofascore.p.rapidapi.com/tournaments/get-matches"
        params = {
            "tournamentId": 17,
            "seasonId": seizoen["id"],
            "pageIndex": page
        }
        
        response = requests.get(url, headers=headers, params=params)
        data = response.json()
        
        wedstrijden = data.get("events", [])
        if not wedstrijden:
            break
        
        for w in wedstrijden:
            alle_wedstrijden.append({
                "event_id": w["id"],
                "season_id": seizoen["id"],
                "date": w["startTimestamp"]
            })
        
        print(f"{seizoen['naam']} - Pagina {page}: {len(wedstrijden)} wedstrijden")
        
        if not data.get("hasNextPage", False):
            break
        page += 1

print(f"\nTotaal: {len(alle_wedstrijden)} wedstrijden opgehaald")

with open("premier_league_wedstrijden.json", "w", encoding="utf-8") as f:
    json.dump(alle_wedstrijden, f, ensure_ascii=False, indent=2)

print("Opgeslagen in premier_league_wedstrijden.json")

Premier League 17/18 - Pagina 0: 30 wedstrijden
Premier League 17/18 - Pagina 1: 30 wedstrijden
Premier League 17/18 - Pagina 2: 30 wedstrijden
Premier League 17/18 - Pagina 3: 30 wedstrijden
Premier League 17/18 - Pagina 4: 30 wedstrijden
Premier League 17/18 - Pagina 5: 30 wedstrijden
Premier League 17/18 - Pagina 6: 30 wedstrijden
Premier League 17/18 - Pagina 7: 30 wedstrijden
Premier League 17/18 - Pagina 8: 30 wedstrijden
Premier League 17/18 - Pagina 9: 30 wedstrijden
Premier League 17/18 - Pagina 10: 30 wedstrijden
Premier League 17/18 - Pagina 11: 30 wedstrijden
Premier League 17/18 - Pagina 12: 27 wedstrijden
Premier League 16/17 - Pagina 0: 30 wedstrijden
Premier League 16/17 - Pagina 1: 30 wedstrijden
Premier League 16/17 - Pagina 2: 30 wedstrijden
Premier League 16/17 - Pagina 3: 30 wedstrijden
Premier League 16/17 - Pagina 4: 30 wedstrijden
Premier League 16/17 - Pagina 5: 30 wedstrijden
Premier League 16/17 - Pagina 6: 30 wedstrijden
Premier League 16/17 - Pagina 7: 30 w

In [2]:
import requests
import json
import time
import csv
import os

headers = {
    'x-rapidapi-key': "2b277abcd2msh0e5627048810020p119057jsn4412b05235c9",
    'x-rapidapi-host': "sofascore.p.rapidapi.com"
}

seizoenen_info = {
    10356: {"naam": "2015-2016", "bestand": "2015-2016"},
    8186:  {"naam": "2014-2015", "bestand": "2014-2015"},
}

match_base  = r"C:\Users\semwi\FPL-Core-Insights\data\Seasonal data\matches"
lineup_base = r"C:\Users\semwi\FPL-Core-Insights\data\Seasonal data\players"

with open('premier_league_wedstrijden.json', 'r', encoding='utf-8') as f:
    all_events = json.load(f)

for season_id, info in seizoenen_info.items():
    season_events = [e for e in all_events if e['season_id'] == season_id]
    print(f"\n=== {info['naam']} | {len(season_events)} wedstrijden ===")

    match_path  = os.path.join(match_base,  f"{info['bestand']}_raw.csv")
    lineup_path = os.path.join(lineup_base, f"{info['bestand']}_players.csv")

    # Check welke events al verwerkt zijn
    already_done = set()
    if os.path.exists(match_path):
        with open(match_path, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row.get('home_team') not in [None, '', 'None']:
                    already_done.add(int(row['match_id']))

    print(f"Al verwerkt: {len(already_done)} | Nog te doen: {len(season_events) - len(already_done)}")

    DELAY = 0.25
    request_count = 0

    for i, event in enumerate(season_events):
        event_id = event['event_id']

        if event_id in already_done:
            print(f"[{i+1}/{len(season_events)}] Skip {event_id}")
            continue

        print(f"[{i+1}/{len(season_events)}] Verwerken event {event_id}...")

        # --- 1. Match details ---
        try:
            res = requests.get(f"https://sofascore.p.rapidapi.com/matches/detail?matchId={event_id}", headers=headers)
            data = res.json()
            if 'message' in data:
                print(f"  QUOTA OP: {data['message']}")
                break
            ed = data.get('event', {})
            request_count += 1
            time.sleep(DELAY)
        except Exception as e:
            print(f"  ERROR details: {e}")
            ed = {}

        # --- 2. Statistics ---
        try:
            res = requests.get(f"https://sofascore.p.rapidapi.com/matches/get-statistics?matchId={event_id}", headers=headers)
            data = res.json()
            if 'message' in data:
                print(f"  QUOTA OP: {data['message']}")
                break
            stats_raw = data.get('statistics', [])
            request_count += 1
            time.sleep(DELAY)
        except Exception as e:
            print(f"  ERROR statistics: {e}")
            stats_raw = []

        # Flatten statistics - alleen ALL periode
        stats_flat = {}
        for period in stats_raw:
            if period.get('period') != 'ALL':
                continue
            for group in period.get('groups', []):
                for item in group.get('statisticsItems', []):
                    key = item.get('key', '')
                    stats_flat[f"home_{key}"] = item.get('homeValue')
                    stats_flat[f"away_{key}"] = item.get('awayValue')

        match_row = {
            'match_id': event_id,
            'season': info['naam'],
            'round': ed.get('roundInfo', {}).get('round'),
            'timestamp': ed.get('startTimestamp'),
            'status': ed.get('status', {}).get('description'),
            'home_team': ed.get('homeTeam', {}).get('name'),
            'away_team': ed.get('awayTeam', {}).get('name'),
            'home_team_id': ed.get('homeTeam', {}).get('id'),
            'away_team_id': ed.get('awayTeam', {}).get('id'),
            'home_goals': ed.get('homeScore', {}).get('current'),
            'away_goals': ed.get('awayScore', {}).get('current'),
            'venue': ed.get('venue', {}).get('name') if ed.get('venue') else None,
            'referee': ed.get('referee', {}).get('name') if ed.get('referee') else None,
            'attendance': ed.get('attendance'),
            **stats_flat
        }

        # --- 3. Lineups ---
        try:
            res = requests.get(f"https://sofascore.p.rapidapi.com/matches/get-lineups?matchId={event_id}", headers=headers)
            data = res.json()
            if 'message' in data:
                print(f"  QUOTA OP: {data['message']}")
                break
            lineups_raw = data
            request_count += 1
            time.sleep(DELAY)
        except Exception as e:
            print(f"  ERROR lineups: {e}")
            lineups_raw = {}

        home_formation = lineups_raw.get('home', {}).get('formation')
        away_formation = lineups_raw.get('away', {}).get('formation')

        lineup_rows = []
        for side in ['home', 'away']:
            team_data = lineups_raw.get(side, {})
            formation = home_formation if side == 'home' else away_formation
            for p in team_data.get('players', []):
                player = p.get('player', {})
                stats = p.get('statistics', {})
                lineup_rows.append({
                    'match_id': event_id,
                    'season': info['naam'],
                    'home_team': ed.get('homeTeam', {}).get('name'),
                    'away_team': ed.get('awayTeam', {}).get('name'),
                    'home_goals': ed.get('homeScore', {}).get('current'),
                    'away_goals': ed.get('awayScore', {}).get('current'),
                    'round': ed.get('roundInfo', {}).get('round'),
                    'timestamp': ed.get('startTimestamp'),
                    'side': side,
                    'formation': formation,
                    'player_id': player.get('id'),
                    'player_name': player.get('name'),
                    'short_name': player.get('shortName'),
                    'position': p.get('position'),
                    'shirt_number': p.get('shirtNumber'),
                    'substitute': p.get('substitute'),
                    'captain': p.get('captain'),
                    'nationality': player.get('country', {}).get('name'),
                    'height': player.get('height'),
                    'market_value': player.get('proposedMarketValueRaw', {}).get('value') if player.get('proposedMarketValueRaw') else None,
                    'rating': stats.get('rating'),
                    'minutes_played': stats.get('minutesPlayed'),
                    'touches': stats.get('touches'),
                    'total_pass': stats.get('totalPass'),
                    'accurate_pass': stats.get('accuratePass'),
                    'total_long_balls': stats.get('totalLongBalls'),
                    'accurate_long_balls': stats.get('accurateLongBalls'),
                    'total_cross': stats.get('totalCross'),
                    'accurate_cross': stats.get('accurateCross'),
                    'key_pass': stats.get('keyPass'),
                    'total_shots': stats.get('totalShots'),
                    'on_target': stats.get('onTargetScoringAttempt'),
                    'shot_off_target': stats.get('shotOffTarget'),
                    'blocked_shot': stats.get('blockedScoringAttempt'),
                    'goals': stats.get('goals'),
                    'goal_assist': stats.get('goalAssist'),
                    'big_chance_created': stats.get('bigChanceCreated'),
                    'big_chance_missed': stats.get('bigChanceMissed'),
                    'hit_woodwork': stats.get('hitWoodwork'),
                    'duel_won': stats.get('duelWon'),
                    'duel_lost': stats.get('duelLost'),
                    'aerial_won': stats.get('aerialWon'),
                    'aerial_lost': stats.get('aerialLost'),
                    'total_tackle': stats.get('totalTackle'),
                    'won_tackle': stats.get('wonTackle'),
                    'interception_won': stats.get('interceptionWon'),
                    'total_clearance': stats.get('totalClearance'),
                    'outfielder_block': stats.get('outfielderBlock'),
                    'total_contest': stats.get('totalContest'),
                    'won_contest': stats.get('wonContest'),
                    'dispossessed': stats.get('dispossessed'),
                    'possession_lost': stats.get('possessionLostCtrl'),
                    'unsuccessful_touch': stats.get('unsuccessfulTouch'),
                    'ball_recovery': stats.get('ballRecovery'),
                    'was_fouled': stats.get('wasFouled'),
                    'fouls': stats.get('fouls'),
                    'total_offside': stats.get('totalOffside'),
                    'penalty_won': stats.get('penaltyWon'),
                    'penalty_conceded': stats.get('penaltyConceded'),
                    'penalty_miss': stats.get('penaltyMiss'),
                    'error_led_to_goal': stats.get('errorLeadToGoal'),
                    'saves': stats.get('saves'),
                    'saves_inside_box': stats.get('savedShotsFromInsideTheBox'),
                    'penalty_save': stats.get('penaltySave'),
                    'punches': stats.get('punches'),
                    'acc_own_half_pass': stats.get('accurateOwnHalfPasses'),
                    'acc_opp_half_pass': stats.get('accurateOppositionHalfPasses'),
                })

        # --- Wegschrijven match rij ---
        fixed_cols = ['match_id', 'season', 'round', 'timestamp', 'status', 'home_team', 'away_team',
                      'home_team_id', 'away_team_id', 'home_goals', 'away_goals', 'venue', 'referee', 'attendance']
        stat_cols = [k for k in sorted(match_row.keys()) if k not in fixed_cols]
        final_cols = fixed_cols + stat_cols

        match_file_exists = os.path.exists(match_path)
        with open(match_path, 'a', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=final_cols, extrasaction='ignore')
            if not match_file_exists:
                writer.writeheader()
            writer.writerow(match_row)

        # --- Wegschrijven lineup rijen ---
        if lineup_rows:
            lineup_file_exists = os.path.exists(lineup_path)
            with open(lineup_path, 'a', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=lineup_rows[0].keys())
                if not lineup_file_exists:
                    writer.writeheader()
                writer.writerows(lineup_rows)

        print(f"  Opgeslagen | Requests: {request_count}")

    print(f"Seizoen {info['naam']} klaar! {request_count} requests gebruikt")

print(f"\nAlles klaar!")


=== 2017-2018 | 387 wedstrijden ===
Al verwerkt: 387 | Nog te doen: 0
[1/387] Skip 7438093
[2/387] Skip 7438038
[3/387] Skip 7438037
[4/387] Skip 7438078
[5/387] Skip 7438071
[6/387] Skip 7438026
[7/387] Skip 7438081
[8/387] Skip 7438095
[9/387] Skip 7438065
[10/387] Skip 7438084
[11/387] Skip 7438044
[12/387] Skip 7438029
[13/387] Skip 7438062
[14/387] Skip 7438057
[15/387] Skip 7769665
[16/387] Skip 7769666
[17/387] Skip 7769669
[18/387] Skip 7769667
[19/387] Skip 7769670
[20/387] Skip 7769668
[21/387] Skip 7438024
[22/387] Skip 7438023
[23/387] Skip 7438080
[24/387] Skip 7438043
[25/387] Skip 7438067
[26/387] Skip 7438076
[27/387] Skip 7438041
[28/387] Skip 7438058
[29/387] Skip 7438055
[30/387] Skip 7438033
[31/387] Skip 7438032
[32/387] Skip 7438052
[33/387] Skip 7438049
[34/387] Skip 7438046
[35/387] Skip 7438074
[36/387] Skip 7438021
[37/387] Skip 7438085
[38/387] Skip 7438030
[39/387] Skip 7438054
[40/387] Skip 7438063
[41/387] Skip 7438064
[42/387] Skip 7438034
[43/387] Skip 

KeyboardInterrupt: 